# Sentiment Analysis Project Overview

This notebook documents the workflow for the airline tweet sentiment analysis project.

The project loads tweet data, cleans and preprocesses the text, compares a VADER rule-based sentiment approach with a RoBERTa transformer model, and evaluates both against the airline-provided sentiment labels.

Key steps include:
- loading and filtering the original tweet dataset
- preprocessing text for VADER analysis
- scoring tweets with VADER on raw and cleaned text
- running RoBERTa sentiment inference via the saved prediction script
- comparing model performance with accuracy, F1, confusion matrices, and classification reports
- checking duplicate and data-quality issues
- exporting a dashboard-ready dataset for further reporting

The final output files are saved under the outputs folder and used for downstream analysis and dashboard reporting.


In [2]:
import pandas as pd

df = pd.read_csv("../data/Tweets.csv")

df = df[["tweet_id", "airline", "airline_sentiment",
         "airline_sentiment_confidence", "negativereason",
         "text", "tweet_created"]]

print(df.shape)
print(df["airline_sentiment"].value_counts())
df.head()

(14640, 7)
airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


,tweet_id,airline,airline_sentiment,airline_sentiment_confidence,negativereason,text,tweet_created
0,570306133677760513,Virgin America,neutral,1.0000,NaN,@VirginAmerica What @dhepburn said.,2015-02-24 11:35:52 -0800
1,570301130888122368,Virgin America,positive,0.3486,NaN,@VirginAmerica plus you've added commercials t...,2015-02-24 11:15:59 -0800
2,570301083672813571,Virgin America,neutral,0.6837,NaN,@VirginAmerica I didn't today... Must mean I n...,2015-02-24 11:15:48 -0800
3,570301031407624196,Virgin America,negative,1.0000,Bad Flight,@VirginAmerica it's really aggressive to blast...,2015-02-24 11:15:36 -0800
4,570300817074462722,Virgin America,negative,1.0000,Can't Tell,@VirginAmerica and it's a really big bad thing...,2015-02-24 11:14:45 -0800


In [3]:
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def clean_tweet(text):
    text = re.sub(r"http\S+", "", text)   # remove links
    text = re.sub(r"@\w+", "", text)      # remove @mentions
    text = re.sub(r"&amp;", "and", text)  # fix HTML leftovers
    text = re.sub(r"\s+", " ", text)      # tidy extra spaces
    return text.strip()

def vader_label(text):
    score = analyzer.polarity_scores(text)["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

df["clean_text"] = df["text"].apply(clean_tweet)
df["vader_raw"] = df["text"].apply(vader_label)
df["vader_clean"] = df["clean_text"].apply(vader_label)

In [4]:
from sklearn.metrics import accuracy_score, classification_report

for col in ["vader_raw", "vader_clean"]:
    print(col, "accuracy:", round(accuracy_score(df["airline_sentiment"], df[col]), 3))

print(classification_report(df["airline_sentiment"], df["vader_clean"]))


vader_raw accuracy: 0.49
vader_clean accuracy: 0.55
              precision    recall  f1-score   support

    negative       0.90      0.51      0.65      9178
     neutral       0.39      0.41      0.40      3099
    positive       0.34      0.89      0.49      2363

    accuracy                           0.55     14640
   macro avg       0.54      0.60      0.51     14640
weighted avg       0.70      0.55      0.57     14640



In [5]:
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = ["negative", "neutral", "positive"]
cm = confusion_matrix(df["airline_sentiment"], df["vader_clean"], labels=labels)
print(pd.DataFrame(cm, index=[f"true_{l}" for l in labels],
                   columns=[f"pred_{l}" for l in labels]))

# negative tweets that VADER got wrong
missed = df[(df["airline_sentiment"] == "negative") & (df["vader_clean"] != "negative")]
pd.set_option("display.max_colwidth", 200)
missed[["clean_text", "vader_clean"]].sample(10, random_state=1)

               pred_negative  pred_neutral  pred_positive
true_negative           4679          1793           2706
true_neutral             445          1267           1387
true_positive             78           179           2106


,clean_text,vader_clean
10152,Now going on 26 minutes on hold.,neutral
5063,want to pick up a customer from sign me up 🙌😏,positive
372,currently in minute 10 of being on hold with cust. service. Do I need to do anything to add a lap child to my reservation?,neutral
280,already contacted Central Baggage and sent DM. is charging me for the dress that was in the lost suitcase #help,positive
3918,submitted customer care form Jan 7th. Still no response. A little longer then 7 - 10 business days,positive
13573,I've been doing this all night. Can someone please help? I need my flt fixed. Please???,positive
11410,that wasn’t my Q but thanks. Wondering why you’re the only ones. I disguised yours to not call you out. ;),positive
2063,- you sure missed the mark on tonight's redeye from LAX to Chicago. What a mess! You can do better!,positive
11233,plz plz plz hold 4438 to EYW. The 810 PHL &gt;CLT is sooo Late Flight. Gonna miss the connection for 's 40th #GTH239,positive
2297,you have a culture issue,neutral


In [6]:
res = pd.read_csv("../outputs/roberta_results.csv")
assert len(res) == len(df) and (res["tweet_id"].values == df["tweet_id"].values).all()
df["roberta"] = res["roberta"].values
df["roberta_score"] = res["roberta_score"].values
print(df["roberta"].value_counts())

roberta
negative    7500
neutral     3991
positive    3149
Name: count, dtype: int64


In [7]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

for col in ["vader_clean", "roberta"]:
    acc = accuracy_score(df["airline_sentiment"], df[col])
    f1 = f1_score(df["airline_sentiment"], df[col], average="macro")
    print(f"{col}: accuracy {acc:.3f}, macro F1 {f1:.3f}")

print(classification_report(df["airline_sentiment"], df["roberta"]))

vader_clean: accuracy 0.550, macro F1 0.514
roberta: accuracy 0.769, macro F1 0.734
              precision    recall  f1-score   support

    negative       0.94      0.77      0.85      9178
     neutral       0.52      0.67      0.59      3099
    positive       0.67      0.90      0.77      2363

    accuracy                           0.77     14640
   macro avg       0.71      0.78      0.73     14640
weighted avg       0.81      0.77      0.78     14640



In [8]:
strong = df[df["airline_sentiment_confidence"] >= 0.7]
print(len(strong), "tweets with label confidence >= 0.7")
for col in ["vader_clean", "roberta"]:
    acc = accuracy_score(strong["airline_sentiment"], strong[col])
    f1 = f1_score(strong["airline_sentiment"], strong[col], average="macro")
    print(f"{col}: accuracy {acc:.3f}, macro F1 {f1:.3f}")
    

10768 tweets with label confidence >= 0.7
vader_clean: accuracy 0.592, macro F1 0.532
roberta: accuracy 0.840, macro F1 0.790


In [9]:
print(df["airline_sentiment"].value_counts(normalize=True).round(3))
print(strong["airline_sentiment"].value_counts(normalize=True).round(3))


airline_sentiment
negative    0.627
neutral     0.212
positive    0.161
Name: proportion, dtype: float64
airline_sentiment
negative    0.701
neutral     0.152
positive    0.147
Name: proportion, dtype: float64


In [10]:
print(classification_report(strong["airline_sentiment"], strong["roberta"]))


              precision    recall  f1-score   support

    negative       0.97      0.84      0.90      7552
     neutral       0.53      0.75      0.62      1634
    positive       0.77      0.96      0.85      1582

    accuracy                           0.84     10768
   macro avg       0.76      0.85      0.79     10768
weighted avg       0.88      0.84      0.85     10768



In [11]:
export = df[["tweet_id", "airline", "airline_sentiment",
             "airline_sentiment_confidence", "negativereason",
             "tweet_created", "text", "vader_clean",
             "roberta", "roberta_score"]].copy()

# simple date column (the first 10 characters are YYYY-MM-DD)
export["date"] = export["tweet_created"].str[:10]

# True/False flags so the dashboard can calculate accuracy easily
export["vader_correct"] = export["vader_clean"] == export["airline_sentiment"]
export["roberta_correct"] = export["roberta"] == export["airline_sentiment"]

export.to_csv("../outputs/dashboard_data.csv", index=False)
print(export.shape)
export.head()

(14640, 13)


,tweet_id,airline,airline_sentiment,airline_sentiment_confidence,negativereason,tweet_created,text,vader_clean,roberta,roberta_score,date,vader_correct,roberta_correct
0,570306133677760513,Virgin America,neutral,1.0000,NaN,2015-02-24 11:35:52 -0800,@VirginAmerica What @dhepburn said.,neutral,neutral,0.940187,2015-02-24,True,True
1,570301130888122368,Virgin America,positive,0.3486,NaN,2015-02-24 11:15:59 -0800,@VirginAmerica plus you've added commercials to the experience... tacky.,neutral,negative,0.888201,2015-02-24,False,False
2,570301083672813571,Virgin America,neutral,0.6837,NaN,2015-02-24 11:15:48 -0800,@VirginAmerica I didn't today... Must mean I need to take another trip!,neutral,neutral,0.666625,2015-02-24,True,True
3,570301031407624196,Virgin America,negative,1.0000,Bad Flight,2015-02-24 11:15:36 -0800,"@VirginAmerica it's really aggressive to blast obnoxious ""entertainment"" in your guests' faces &amp; they have little recourse",negative,negative,0.887786,2015-02-24,True,True
4,570300817074462722,Virgin America,negative,1.0000,Can't Tell,2015-02-24 11:14:45 -0800,@VirginAmerica and it's a really big bad thing about it,negative,negative,0.935381,2015-02-24,True,True


In [12]:
extra = df["tweet_id"].duplicated().sum()
dups = df[df["tweet_id"].duplicated(keep=False)]
print(extra, "extra rows;", dups["tweet_id"].nunique(), "tweet_ids affected")
print(df.duplicated().sum(), "rows identical across every column")
print(dups.groupby("tweet_id")["airline_sentiment"].nunique().gt(1).sum(),
      "duplicated ids with conflicting labels")

155 extra rows; 155 tweet_ids affected
62 rows identical across every column
18 duplicated ids with conflicting labels


In [13]:
from sklearn.metrics import accuracy_score, f1_score

dedup = df.drop_duplicates(subset="tweet_id")
print(len(dedup), "tweets after removing repeats")
for col in ["vader_clean", "roberta"]:
    acc = accuracy_score(dedup["airline_sentiment"], dedup[col])
    f1 = f1_score(dedup["airline_sentiment"], dedup[col], average="macro")
    print(f"{col}: accuracy {acc:.3f}, macro F1 {f1:.3f}")

14485 tweets after removing repeats
vader_clean: accuracy 0.550, macro F1 0.514
roberta: accuracy 0.769, macro F1 0.734


In [14]:
print(df.groupby("airline")["airline_sentiment"]
        .apply(lambda s: (s == "negative").mean()).round(3).sort_values(ascending=False))
print(df[df["airline_sentiment"] == "negative"]["negativereason"].value_counts())
print(df["airline"].value_counts())

airline
US Airways        0.777
American          0.710
United            0.689
Southwest         0.490
Delta             0.430
Virgin America    0.359
Name: airline_sentiment, dtype: float64
negativereason
Customer Service Issue         2910
Late Flight                    1665
Can't Tell                     1190
Cancelled Flight                847
Lost Luggage                    724
Bad Flight                      580
Flight Booking Problems         529
Flight Attendant Complaints     481
longlines                       178
Damaged Luggage                  74
Name: count, dtype: int64
airline
United            3822
US Airways        2913
American          2759
Southwest         2420
Delta             2222
Virgin America     504
Name: count, dtype: int64
